# IndicSent-M — Explainable AI (XAI) & Live Meme Demo (Colab)

Completes the remaining technical deliverables using your trained checkpoints saved in Google Drive (`/content/drive/MyDrive/IndicSent-M_Final`):

1. **Late Fusion ($\alpha = 0.55$) Confusion Matrix & Full Test Export** — saves `cm_late_fusion.png` and `test_preds_late_fusion.csv` for your research paper.
2. **Dual-Modality Explainable AI (XAI):**
   - **Visual XAI:** Class-discriminative **ViT Grad-CAM** heatmaps (with automatic Attention Rollout fallback) highlighting which facial expression or image region drove the sentiment prediction.
   - **Textual XAI:** Word-level **Leave-One-Out Occlusion Attribution** on MuRIL showing which Hindi/Hinglish words pushed the sentiment score up or down.
3. **Paper-Ready Case Studies:** Automatically finds test memes where **Text-only failed** and **Multimodal Late Fusion succeeded**, saving 6 high-resolution 3-panel figures to Google Drive (`xai_figures/`).
4. **End-to-End Live Meme Demo (Gradio + EasyOCR):** Upload any raw Indian-language meme image (Hinglish, Hindi/Devanagari, or English) to get live OCR extraction, unimodal vs. multimodal predictions, Grad-CAM heatmap, and highlighted word importance.

In [ ]:
!pip install -q transformers emoji gradio easyocr scikit-learn pandas pillow seaborn matplotlib

import os, glob, re, json, html, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
import emoji
from PIL import Image, ImageFile
from torchvision import transforms
from transformers import AutoTokenizer, AutoModel, ViTModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from google.colab import drive

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/IndicSent-M_Final"
XAI_DIR  = f"{SAVE_DIR}/xai_figures"
os.makedirs(XAI_DIR, exist_ok=True)

# Unzip Memotion 3.0 locally if running in a fresh Colab session so test image paths resolve
DATA_ROOT = "/content/memotion_data"
if not os.path.exists(DATA_ROOT) or not os.listdir(DATA_ROOT):
    zip_candidates = [
        "/content/IndicSent-M.zip",
        "/content/drive/MyDrive/IndicSent-M.zip",
    ] + glob.glob("/content/drive/MyDrive/**/IndicSent-M.zip", recursive=True)
    ZIP_PATH = next((p for p in zip_candidates if os.path.exists(p)), None)
    if ZIP_PATH:
        os.makedirs(DATA_ROOT, exist_ok=True)
        print("Unzipping dataset from:", ZIP_PATH)
        !unzip -q "{ZIP_PATH}" -d "{DATA_ROOT}"
    else:
        print("⚠️ IndicSent-M.zip not found. Cell 4 test-set figures need test images, but Cell 5 Gradio demo works on any uploaded meme!")

print("Artifacts in SAVE_DIR:", sorted(os.listdir(SAVE_DIR)))

In [ ]:
with open(f"{SAVE_DIR}/training_config.json", "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

TEXT_MODEL  = CONFIG["text_model"]
IMAGE_MODEL = CONFIG["image_model"]
MAX_LEN     = CONFIG["max_len"]
CLASS_NAMES = CONFIG["class_names"]
LATE_ALPHA  = 0.55  # Optimal validation-tuned weight (55% Text MuRIL + 45% Image ViT)

# Load saved tokenizer from Drive (fallback to Hugging Face Hub if missing)
tok_path = f"{SAVE_DIR}/tokenizer"
tokenizer = AutoTokenizer.from_pretrained(tok_path if os.path.exists(tok_path) else TEXT_MODEL)

eval_img_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

def clean_text(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    value = emoji.demojize(str(value), delimiters=(" ", " "))
    value = re.sub(r"http\S+|www\.\S+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

class IndicSentModel(nn.Module):
    def __init__(self, mode="multimodal", num_classes=3, dropout=0.3):
        super().__init__()
        self.mode = mode

        if mode in ("text", "multimodal"):
            self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL)
            text_dim = self.text_encoder.config.hidden_size

        if mode in ("image", "multimodal"):
            # Use eager attention so output_attentions=True works cleanly for Attention Rollout fallback
            try:
                self.image_encoder = ViTModel.from_pretrained(IMAGE_MODEL, attn_implementation="eager")
            except TypeError:
                self.image_encoder = ViTModel.from_pretrained(IMAGE_MODEL)
            image_dim = self.image_encoder.config.hidden_size

        if mode == "text":
            self.classifier = nn.Sequential(
                nn.Dropout(dropout), nn.Linear(text_dim, 256), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(256, num_classes)
            )
        elif mode == "image":
            self.classifier = nn.Sequential(
                nn.Dropout(dropout), nn.Linear(image_dim, 256), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(256, num_classes)
            )
        else:
            self.text_proj  = nn.Sequential(nn.Linear(text_dim, 256), nn.LayerNorm(256), nn.GELU())
            self.image_proj = nn.Sequential(nn.Linear(image_dim, 256), nn.LayerNorm(256), nn.GELU())
            self.gate       = nn.Sequential(nn.Linear(512, 256), nn.Sigmoid())
            self.classifier = nn.Sequential(
                nn.LayerNorm(1024), nn.Dropout(dropout),
                nn.Linear(1024, 256), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(256, num_classes)
            )

    def encode_text(self, input_ids, attention_mask):
        hidden = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).type_as(hidden)
        return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)

    def encode_image(self, pixel_values):
        return self.image_encoder(pixel_values=pixel_values).last_hidden_state[:, 0]

    def forward(self, input_ids, attention_mask, pixel_values):
        if self.mode == "text":
            return self.classifier(self.encode_text(input_ids, attention_mask))
        if self.mode == "image":
            return self.classifier(self.encode_image(pixel_values))
        t = self.text_proj(self.encode_text(input_ids, attention_mask))
        v = self.image_proj(self.encode_image(pixel_values))
        gate = self.gate(torch.cat([t, v], dim=-1))
        fused = torch.cat([gate * t + (1.0 - gate) * v, t, v, torch.abs(t - v)], dim=-1)
        return self.classifier(fused)

def load_trained_model(mode):
    ckpt = f"{SAVE_DIR}/best_{mode}.pt"
    assert os.path.exists(ckpt), f"Checkpoint missing: {ckpt}"
    m = IndicSentModel(mode=mode).to(DEVICE)
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    m.eval()
    return m

text_model  = load_trained_model("text")
image_model = load_trained_model("image")
mm_model    = load_trained_model("multimodal")
print("✅ Loaded all 3 trained models (MuRIL Text, ViT Image, Gated Multimodal) successfully!")

In [ ]:
# --- Dual-Modality Inference & Explainable AI (XAI) Engine ---

@torch.no_grad()
def predict_all_modalities(pil_img, raw_text, alpha=LATE_ALPHA):
    cleaned = clean_text(raw_text)
    enc = tokenizer(
        cleaned, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt"
    )
    ids    = enc["input_ids"].to(DEVICE)
    mask   = enc["attention_mask"].to(DEVICE)
    pixels = eval_img_tf(pil_img.convert("RGB")).unsqueeze(0).to(DEVICE)

    p_text  = F.softmax(text_model(ids, mask, pixels), dim=-1)[0].cpu().numpy()
    p_image = F.softmax(image_model(ids, mask, pixels), dim=-1)[0].cpu().numpy()
    p_gated = F.softmax(mm_model(ids, mask, pixels), dim=-1)[0].cpu().numpy()
    p_late  = alpha * p_text + (1.0 - alpha) * p_image

    return {
        "cleaned_text": cleaned,
        "text_probs": p_text,
        "image_probs": p_image,
        "gated_probs": p_gated,
        "late_probs": p_late,
    }


def compute_vit_gradcam(pil_img, target_class, model_for_cam=None, raw_text=""):
    """
    Class-discriminative ViT Grad-CAM on the final Transformer block (14x14 patch grid).
    Automatically falls back to ViT Attention Rollout if all patch gradients are <= 0.
    """
    cam_model = image_model if model_for_cam is None else model_for_cam
    cam_model.eval()
    cam_model.zero_grad()

    rgb_img = pil_img.convert("RGB")
    pixels = eval_img_tf(rgb_img).unsqueeze(0).to(DEVICE)
    pixels.requires_grad_(True)

    enc = tokenizer(
        clean_text(raw_text), truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt"
    )
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)

    activations = {}
    gradients = {}

    # Hook final ViT block LayerNorm output ([B, 197, 768])
    target_layer = cam_model.image_encoder.encoder.layer[-1].layernorm_before

    def fwd_hook(module, inp, out):
        activations["val"] = out
        if out.requires_grad:
            out.register_hook(lambda g: gradients.setdefault("val", g))

    handle = target_layer.register_forward_hook(fwd_hook)
    try:
        logits = cam_model(ids, mask, pixels)
        score = logits[0, int(target_class)]
        score.backward()
    finally:
        handle.remove()

    # Build 14x14 Grad-CAM map from patch tokens (skipping [CLS] at index 0)
    cam_map = None
    if "val" in activations and "val" in gradients:
        act  = activations["val"][0, 1:, :].detach()   # [196, 768]
        grad = gradients["val"][0, 1:, :].detach()     # [196, 768]
        weights = grad.mean(dim=0)                     # [768]
        cam = F.relu((act * weights).sum(dim=-1))      # [196]
        if cam.max().item() > 1e-8:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
            grid = int(np.sqrt(cam.numel()))
            cam_map = cam.reshape(grid, grid).cpu().numpy()

    # Fallback: Attention Rollout across all ViT layers if Grad-CAM map is flat
    if cam_map is None:
        with torch.no_grad():
            vit_out = cam_model.image_encoder(pixel_values=pixels.detach(), output_attentions=True)
            attns = [a for a in (vit_out.attentions or []) if a is not None]
            if attns:
                rollout = torch.eye(attns[0].size(-1), device=DEVICE)
                for a in attns:
                    a_fused = a[0].mean(dim=0)
                    a_fused = a_fused + torch.eye(a_fused.size(-1), device=DEVICE)
                    a_fused = a_fused / a_fused.sum(dim=-1, keepdim=True)
                    rollout = torch.matmul(a_fused, rollout)
                cls_attn = rollout[0, 1:]
                cls_attn = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min() + 1e-8)
                grid = int(np.sqrt(cls_attn.numel()))
                cam_map = cls_attn.reshape(grid, grid).cpu().numpy()
            else:
                cam_map = np.zeros((14, 14), dtype=np.float32)

    cam_model.zero_grad()

    # Resize heatmap smoothly to original meme dimensions and blend
    w, h = rgb_img.size
    heat_pil = Image.fromarray((cam_map * 255).astype(np.uint8)).resize((w, h), resample=Image.BILINEAR)
    heat_norm = np.asarray(heat_pil, dtype=np.float32) / 255.0
    colored = (mpl_cm.jet(heat_norm)[..., :3] * 255).astype(np.uint8)

    orig_np = np.asarray(rgb_img, dtype=np.float32)
    blended = np.clip(0.55 * orig_np + 0.45 * colored.astype(np.float32), 0, 255).astype(np.uint8)
    return Image.fromarray(blended)


@torch.no_grad()
def compute_word_importance(raw_text, target_class, max_words=45):
    """
    Word-level Leave-One-Out Occlusion Attribution on MuRIL.
    Score > 0: word supports the predicted sentiment; Score < 0: word opposes it.
    Works cleanly across Hinglish, Devanagari Hindi, English, and emojis.
    """
    cleaned = clean_text(raw_text)
    words = cleaned.split()[:max_words]
    if not words:
        return []

    variants = [" ".join(words)] + [" ".join(words[:i] + words[i+1:]) for i in range(len(words))]
    enc = tokenizer(
        variants, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt"
    )
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)
    dummy_px = torch.zeros((len(variants), 3, 224, 224), device=DEVICE)

    probs = F.softmax(text_model(ids, mask, dummy_px), dim=-1)[:, int(target_class)].cpu().numpy()
    base_prob = probs[0]
    deltas = base_prob - probs[1:]  # positive = removing this word lowers target probability
    return list(zip(words, deltas.tolist()))


def render_word_html(word_scores, pred_label):
    """Renders Hindi/Hinglish word attributions as clean color-coded HTML spans for Gradio."""
    if not word_scores:
        return "<p style='color:#666;'>No text detected in this meme (prediction relies on visual cues).</p>"

    max_abs = max(abs(s) for _, s in word_scores) + 1e-6
    spans = []
    for w, s in word_scores:
        norm = s / max_abs
        alpha = min(0.85, 0.12 + 0.73 * abs(norm))
        bg = f"rgba(34, 197, 94, {alpha:.2f})" if norm >= 0 else f"rgba(239, 68, 68, {alpha:.2f})"
        border = "#15803d" if norm >= 0 else "#b91c1c"
        spans.append(
            f"<span title='Impact on {pred_label}: {s:+.4f}' "
            f"style='display:inline-block; margin:3px; padding:3px 7px; border-radius:6px; "
            f"background:{bg}; border:1px solid {border}; color:#111; font-size:15px;'>"
            f"{html.escape(w)} <small style='opacity:0.75;'>({s:+.2f})</small></span>"
        )
    legend = (
        f"<div style='margin-bottom:8px; font-size:13px; color:#374151;'>"
        f"<b>Word Attribution for {pred_label}:</b> "
        f"<span style='background:rgba(34,197,94,0.45); padding:2px 6px; border-radius:4px;'>Green = Supports {pred_label}</span> &nbsp;"
        f"<span style='background:rgba(239,68,68,0.45); padding:2px 6px; border-radius:4px;'>Red = Opposes {pred_label}</span></div>"
    )
    return legend + "<div style='line-height:2.1;'>" + "".join(spans) + "</div>"

print("✅ Dual-Modality XAI Engine (ViT Grad-CAM + MuRIL Word Attribution) ready!")

In [ ]:
# --- Paper Deliverables: Late Fusion Confusion Matrix + 6 XAI Case Study Figures ---

preds_path = f"{SAVE_DIR}/test_preds_all.csv"
df_test = pd.read_csv(preds_path)

# Compute Late Fusion (alpha=0.55) posteriors & predictions on the full test set
p_text_all  = df_test[["prob_neg_text",  "prob_neu_text",  "prob_pos_text"]].values
p_image_all = df_test[["prob_neg_image", "prob_neu_image", "prob_pos_image"]].values
p_late_all  = LATE_ALPHA * p_text_all + (1.0 - LATE_ALPHA) * p_image_all

df_test["pred_late"]     = p_late_all.argmax(axis=1)
df_test["prob_neg_late"] = p_late_all[:, 0].round(4)
df_test["prob_neu_late"] = p_late_all[:, 1].round(4)
df_test["prob_pos_late"] = p_late_all[:, 2].round(4)
df_test["conf_late"]     = p_late_all.max(axis=1).round(4)
df_test.to_csv(f"{SAVE_DIR}/test_preds_late_fusion.csv", index=False)

print(f"📊 TEST REPORT — MULTIMODAL LATE FUSION (alpha={LATE_ALPHA}):")
print(classification_report(df_test["label"], df_test["pred_late"], target_names=CLASS_NAMES, digits=4))

# Save Late Fusion Confusion Matrix to Drive
cm_late = confusion_matrix(df_test["label"], df_test["pred_late"], labels=[0, 1, 2])
plt.figure(figsize=(6, 5))
sns.heatmap(cm_late, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title(f"Confusion Matrix — MULTIMODAL LATE FUSION (α={LATE_ALPHA})")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/cm_late_fusion.png", dpi=220)
plt.show()

# Build filename lookup in case local paths shifted across Colab sessions
local_img_index = {}
if os.path.exists(DATA_ROOT):
    for r, _, fs in os.walk(DATA_ROOT):
        for f in fs:
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                local_img_index[f] = os.path.join(r, f)

def resolve_test_img(path_str):
    if isinstance(path_str, str) and os.path.exists(path_str):
        return path_str
    fname = os.path.basename(str(path_str))
    return local_img_index.get(fname, None)

df_test["resolved_path"] = df_test["image_path"].apply(resolve_test_img)

# Select up to 6 diverse test memes (2 per sentiment class) where Multimodal Late Fusion FIXED Text-only mistakes
wins = df_test[
    (df_test["pred_late"] == df_test["label"]) &
    (df_test["pred_text"] != df_test["label"]) &
    (df_test["resolved_path"].notna()) &
    (df_test["text"].astype(str).str.split().str.len() >= 4)
].sort_values("conf_late", ascending=False)

print(f"\n🔎 Found {len(wins)} memes where Multimodal Late Fusion corrected Text-only errors!")

selected_rows = []
for cls_idx in [0, 1, 2]:
    cls_wins = wins[wins["label"] == cls_idx].head(2)
    selected_rows.append(cls_wins)
selected_df = pd.concat(selected_rows).reset_index(drop=True)

for idx, row in selected_df.iterrows():
    pil_img = Image.open(row["resolved_path"]).convert("RGB")
    gt_lbl   = CLASS_NAMES[int(row["label"])]
    txt_lbl  = CLASS_NAMES[int(row["pred_text"])]
    img_lbl  = CLASS_NAMES[int(row["pred_image"])]
    late_lbl = CLASS_NAMES[int(row["pred_late"])]

    # Generate Visual Grad-CAM & Textual Occlusion explanations
    cam_overlay = compute_vit_gradcam(pil_img, target_class=int(row["pred_late"]), model_for_cam=image_model, raw_text=row["text"])
    word_scores = compute_word_importance(row["text"], target_class=int(row["pred_late"]))

    # Top 8 most impactful words by absolute magnitude
    top_words = sorted(word_scores, key=lambda x: abs(x[1]), reverse=True)[:8][::-1]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(
        f"Case {idx+1} | Ground Truth: {gt_lbl}  |  Text-Only: {txt_lbl} (✗)  |  Image-Only: {img_lbl}  |  Multimodal Late Fusion: {late_lbl} (✓, {row['conf_late']*100:.1f}%)",
        fontsize=12, fontweight="bold", y=1.02
    )

    axes[0].imshow(pil_img)
    axes[0].set_title("Original Meme", fontsize=11)
    axes[0].axis("off")

    axes[1].imshow(cam_overlay)
    axes[1].set_title(f"ViT Grad-CAM (Visual Focus for {late_lbl})", fontsize=11)
    axes[1].axis("off")

    if top_words:
        w_labels = [w for w, _ in top_words]
        w_vals   = [v for _, v in top_words]
        colors   = ["#16a34a" if v >= 0 else "#dc2626" for v in w_vals]
        axes[2].barh(w_labels, w_vals, color=colors)
        axes[2].axvline(0, color="#333", linewidth=0.8)
        axes[2].set_title(f"MuRIL Word Attribution (Δ Prob for {late_lbl})", fontsize=11)
        axes[2].set_xlabel("Impact (+ supports, - opposes)")
        axes[2].grid(axis="x", alpha=0.3)
    else:
        axes[2].text(0.5, 0.5, "No OCR tokens", ha="center", va="center")
        axes[2].axis("off")

    plt.tight_layout()
    out_file = f"{XAI_DIR}/xai_case_{idx+1}_{gt_lbl.lower()}.png"
    plt.savefig(out_file, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"💾 Saved {out_file} | OCR: {str(row['text'])[:140]}\n")

In [ ]:
# --- Live End-to-End Indian Meme Sentiment & XAI Demo (EasyOCR + Gradio) ---
import easyocr
import gradio as gr

print("Initializing EasyOCR (English + Devanagari Hindi)...")
ocr_reader = easyocr.Reader(["en", "hi"], gpu=(DEVICE.type == "cuda"))
print("✅ EasyOCR ready!")

def analyze_meme(image, custom_text, fusion_mode, alpha_weight):
    if image is None:
        return None, None, "", None, "<p>Please upload a meme image first.</p>"

    pil_img = image.convert("RGB")

    # Run EasyOCR automatically unless the user typed/edited custom text
    if custom_text and custom_text.strip():
        extracted_text = custom_text.strip()
    else:
        ocr_lines = ocr_reader.readtext(np.array(pil_img), detail=0, paragraph=True)
        extracted_text = " ".join(ocr_lines).strip()

    preds = predict_all_modalities(pil_img, extracted_text, alpha=float(alpha_weight))

    if fusion_mode.startswith("Late Fusion"):
        final_probs = preds["late_probs"]
        cam_backbone = image_model
    else:
        final_probs = preds["gated_probs"]
        cam_backbone = mm_model

    pred_idx = int(np.argmax(final_probs))
    pred_lbl = CLASS_NAMES[pred_idx]

    # 1. Confidence dictionary for gr.Label
    label_dict = {CLASS_NAMES[i]: float(final_probs[i]) for i in range(3)}

    # 2. Modality Comparison Table (great for viva demonstration!)
    def fmt_row(name, probs):
        best_i = int(np.argmax(probs))
        return [
            name,
            CLASS_NAMES[best_i],
            f"{probs[0]*100:.1f}%",
            f"{probs[1]*100:.1f}%",
            f"{probs[2]*100:.1f}%",
        ]

    breakdown_df = pd.DataFrame(
        [
            fmt_row("Text-Only (MuRIL)", preds["text_probs"]),
            fmt_row("Image-Only (ViT)", preds["image_probs"]),
            fmt_row("Multimodal (Gated Fusion)", preds["gated_probs"]),
            fmt_row(f"Multimodal (Late Fusion, α={alpha_weight:.2f})", preds["late_probs"]),
        ],
        columns=["Model", "Prediction", "Negative", "Neutral", "Positive"],
    )

    # 3. Dual-Modality XAI
    heatmap_img = compute_vit_gradcam(pil_img, target_class=pred_idx, model_for_cam=cam_backbone, raw_text=extracted_text)
    word_scores = compute_word_importance(extracted_text, target_class=pred_idx)
    word_html   = render_word_html(word_scores, pred_lbl)

    return label_dict, breakdown_df, extracted_text, heatmap_img, word_html

with gr.Blocks(title="IndicSent-M: Multimodal Indian Meme Sentiment & XAI") as demo:
    gr.Markdown(
        "# 🇮🇳 IndicSent-M: Multimodal Sentiment Analysis for Indian Memes\n"
        "Upload any Indian-language meme (Hinglish, Hindi/Devanagari, or English). "
        "The system automatically runs **EasyOCR**, predicts sentiment via **MuRIL + ViT Multimodal Fusion**, "
        "and explains the decision with **ViT Grad-CAM** and **word-level attribution**."
    )

    with gr.Row():
        with gr.Column(scale=1):
            img_in = gr.Image(type="pil", label="Upload Meme Image")
            txt_in = gr.Textbox(
                label="Optional Text Override (leave blank to auto-extract via EasyOCR)",
                placeholder="Leave empty for automatic EasyOCR extraction...",
                lines=2,
            )
            fusion_in = gr.Radio(
                choices=["Late Fusion (Best Macro-F1)", "Gated Fusion (Best Accuracy)"],
                value="Late Fusion (Best Macro-F1)",
                label="Multimodal Fusion Strategy",
            )
            alpha_in = gr.Slider(
                minimum=0.0, maximum=1.0, value=0.55, step=0.05,
                label="Late Fusion Text Weight (α) — 0.55 optimal on Memotion 3.0",
            )
            run_btn = gr.Button("Analyze Meme Sentiment & Explain", variant="primary")

        with gr.Column(scale=1):
            pred_out = gr.Label(num_top_classes=3, label="Final Multimodal Sentiment Prediction")
            ocr_out  = gr.Textbox(label="Extracted / Analyzed Meme Text", lines=2)
            tbl_out  = gr.Dataframe(label="Unimodal vs. Multimodal Breakdown", interactive=False)

    with gr.Row():
        cam_out  = gr.Image(type="pil", label="Visual Explanation — ViT Grad-CAM Heatmap")
        html_out = gr.HTML(label="Textual Explanation — MuRIL Word Importance")

    run_btn.click(
        fn=analyze_meme,
        inputs=[img_in, txt_in, fusion_in, alpha_in],
        outputs=[pred_out, tbl_out, ocr_out, cam_out, html_out],
    )

demo.launch(share=True, debug=False)

## Viva & Paper Tips for Using These Outputs

- **Research Paper Figures:** Download `cm_late_fusion.png` and your best 3–4 figures from `/content/drive/MyDrive/IndicSent-M_Final/xai_figures/` directly from Google Drive.
- **Live Viva Demo:** When Cell 5 finishes running, click the public `https://*.gradio.live` link printed at the bottom to show live predictions on new Indian memes right in your browser (valid for 72 hours per Colab session).
- **Sarcasm Showcase:** Upload a sarcastic Hinglish meme (positive caption + frustrated reaction image) and point out the **Unimodal vs. Multimodal Breakdown** table alongside the **Grad-CAM** face highlight and **red/green word spans**.